# Iftixor — Google Colab orqali GPU bilan o'qitish

Bu notebook Iftixor modelini bepul Colab GPU'sida tez o'qitish uchun.

**Avval yoqing:** Yuqoridagi menyudan `Runtime -> Change runtime type -> T4 GPU` ni tanlang, so'ng har bir katakchani tepadan pastga qarab ishga tushiring (Shift+Enter).

In [ ]:
import torch
print("GPU mavjud:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU nomi:", torch.cuda.get_device_name(0))
else:
    print("GPU topilmadi — Runtime -> Change runtime type -> T4 GPU ni tanlab, qayta ishga tushiring.")

## 1. Repozitoriyani yuklab olish

In [ ]:
!git clone https://github.com/Iftix0r/Iftixor.git
%cd Iftixor
!pip install -q python-dotenv

## 2. (Ixtiyoriy) Serveringizdan `conversations.txt` yoki boshqa qo'shimcha ma'lumot yuklash

Agar serveringizdagi `data/conversations.txt` yoki boshqa matn faylini shu yerda ham qo'shib o'qitmoqchi bo'lsangiz, uni avval o'z kompyuteringizga yuklab oling (`scp` bilan), so'ng shu katakchani ishga tushirib, faylni tanlang. Agar kerak bo'lmasa, bu katakchani o'tkazib yuboring.

In [ ]:
from google.colab import files
import shutil

uploaded = files.upload()
for fname in uploaded.keys():
    shutil.move(fname, f"data/{fname}")
    print(f"data/{fname} ga joylandi")

## 3. O'qitish (GPU bilan)

GPU bo'lgani uchun `--batch-size`ni kattaroq, `--steps`ni ham ko'proq qilish mumkin — CPU'dagi soatlab vaqt o'rniga bu bir necha daqiqada tugaydi.

Agar 2-qadamda qo'shimcha fayl yuklagan bo'lsangiz, uni pastdagi `--data` ro'yxatiga qo'shing (masalan `data/conversations.txt`).

In [ ]:
!python -m iftixor.train \
  --data data/corpus.txt data/synthetic.txt \
  --steps 5000 \
  --batch-size 64 \
  --device auto \
  --out checkpoints/iftixor.pt

## 4. Terminalda tez sinash (ixtiyoriy)

In [ ]:
from iftixor.generate import load_model, generate_text

model, tokenizer = load_model("checkpoints/iftixor.pt")
prompt = "Foydalanuvchi: Salom!\nIftixor:"
print(generate_text(model, tokenizer, prompt, max_new_tokens=100))

## 5. Tayyor checkpointni yuklab olish

Bu faylni kompyuteringizga saqlab, so'ng `scp` yoki SFTP orqali serveringizdagi `checkpoints/iftixor.pt` fayli o'rniga joylashtiring, keyin serverda botni qayta ishga tushiring (`systemctl restart iftixor`) yoki Telegram'da `/reload` buyrug'ini yuboring.

In [ ]:
from google.colab import files
files.download("checkpoints/iftixor.pt")